# 04 – Evaluation: verification, unknown faces, and errors

Notebook 03 assumed everyone is in the gallery. A real system also meets **strangers**, and it
must decide *how similar is similar enough*. That decision is a **threshold** on cosine similarity.

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import src...` works from the notebooks folder

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import load_lfw, open_set_split, split
from src.matching import UNKNOWN, build_gallery, identify
from src.metrics import all_pairs, open_set_metrics, verification_metrics
from src.train import feature_sets, gallery_threshold

sns.set_theme(style="whitegrid")
images, y, names = load_lfw()
train_idx, test_idx = split(y)
feats = feature_sets(images, train_idx)

## 1. Verification: "are these two photos the same person?"

Every pair of test images is scored. Metrics:
- **AUC:** 1.0 = perfect separation.
- **EER (equal error rate):** the threshold where false accepts = false rejects. Lower is better.
- **TAR@FAR=1%:** the share of genuine pairs accepted when only 1% of impostor pairs get through.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
rows = []
for fname, F in feats.items():
    m = verification_metrics(*all_pairs(F[test_idx], y[test_idx]))
    fpr, tpr, _ = m["roc"]
    ax.plot(fpr, tpr, label=f"{fname} (AUC {m['auc']:.3f})")
    rows.append({"features": fname, **{k: v for k, v in m.items() if k != "roc"}})
ax.set_xscale("log")
ax.set_xlim(1e-4, 1)
ax.set_xlabel("False accept rate (log scale)")
ax.set_ylabel("True accept rate")
ax.set_title("ROC: face verification on LFW test pairs")
ax.legend()
plt.tight_layout()
plt.show()
pd.DataFrame(rows).round(3)

## 2. Open-set identification: recognising known people *and* rejecting strangers

We hide 12 people completely (they play "strangers"). The gallery holds the training images of
the other 50 people. The probes are the known people's test images plus every image of the
strangers.

We compare two ways to pick the threshold, both aiming at 1% false accepts and both using
**training data only**:

1. **Pair threshold:** from photo-vs-photo similarities, as in section 1.
2. **Gallery threshold:** from photo-vs-*average face* similarities (`gallery_threshold` in
   `src/train.py`). This matches how the gallery is actually used, and it is what the webcam
   demo uses.

In [ ]:
MODEL = "ArcFace"
F = feats[MODEL]
known_ids, unknown_ids = open_set_split(y)

gallery_idx = train_idx[np.isin(y[train_idx], known_ids)]
probe_idx = np.concatenate([test_idx[np.isin(y[test_idx], known_ids)],
                            np.flatnonzero(np.isin(y, unknown_ids))])
ids, centroids = build_gallery(F[gallery_idx], y[gallery_idx])

pair_thr = verification_metrics(*all_pairs(F[gallery_idx], y[gallery_idx]))["threshold@far=0.01"]
gallery_thr = gallery_threshold(F[gallery_idx], y[gallery_idx])
print(f"Gallery: {len(ids)} people from {len(gallery_idx)} images")
print(f"Probes:  {np.isin(y[probe_idx], known_ids).sum()} known + {np.isin(y[probe_idx], unknown_ids).sum()} stranger images")
print(f"Pair threshold: {pair_thr:.3f}   Gallery threshold: {gallery_thr:.3f}")

In [ ]:
sweep = pd.DataFrame([
    {"threshold": t, **open_set_metrics(y[probe_idx], identify(F[probe_idx], ids, centroids, t)[0], known_ids)}
    for t in np.linspace(0, 0.8, 81)
])

fig, ax = plt.subplots(figsize=(9, 4))
for col, label in [("known_accuracy", "Known people named correctly"),
                   ("false_reject_rate", "Known people called 'unknown'"),
                   ("false_accept_rate", "Strangers given a name")]:
    ax.plot(sweep["threshold"], sweep[col], label=label)
ax.axvline(pair_thr, color="grey", ls=":", label=f"pair threshold ({pair_thr:.2f})")
ax.axvline(gallery_thr, color="black", ls="--", label=f"gallery threshold ({gallery_thr:.2f})")
ax.set_xlabel("Cosine-similarity threshold")
ax.set_ylabel("Rate")
ax.set_title(f"Open-set identification with {MODEL}: the threshold trade-off")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

pd.DataFrame({name: open_set_metrics(y[probe_idx], identify(F[probe_idx], ids, centroids, t)[0], known_ids)
              for name, t in [("pair threshold", pair_thr), ("gallery threshold", gallery_thr)]}).round(3)

Similarity to a person's **average** face is higher than to a single photo, for strangers too.
So the pair threshold lets many strangers through. A threshold must be tuned on the same kind
of comparison the system actually makes. This is a good point to discuss in your report.

## 3. Error analysis: the hardest pairs

In [ ]:
scores, same = all_pairs(F[test_idx], y[test_idx])
I, J = np.triu_indices(len(test_idx), k=1)

def show_pairs(pair_ids, title):
    fig, axes = plt.subplots(2, len(pair_ids), figsize=(2.2 * len(pair_ids), 5))
    for col, p in enumerate(pair_ids):
        for row, k in enumerate((I[p], J[p])):
            i = test_idx[k]
            axes[row, col].imshow(images[i])
            axes[row, col].set_title(names[y[i]].split()[-1], fontsize=9)
            axes[row, col].axis("off")
        axes[0, col].set_title(f"sim {scores[p]:.2f}\n" + names[y[test_idx[I[p]]]].split()[-1], fontsize=9)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

genuine, impostor = np.flatnonzero(same == 1), np.flatnonzero(same == 0)
show_pairs(genuine[np.argsort(scores[genuine])[:6]], "Same person, LOWEST similarity (would be falsely rejected)")
show_pairs(impostor[np.argsort(scores[impostor])[-6:][::-1]], "Different people, HIGHEST similarity (would be falsely accepted)")

## 4. Limitations and ethics (for the report)

- **Dataset bias:** LFW is mostly photos of male, light-skinned public figures from the 2000s.
  Accuracy here says little about performance on other groups. LFW has no demographic labels, so
  a fairness analysis needs another dataset (e.g. RFW or FairFace).
- **Easy conditions:** LFW photos are mostly frontal press photos. A webcam in poor light is harder.
- **Pretrained models:** ArcFace and Facenet were trained on large web-scraped face datasets,
  which raises its own consent questions.
- **GDPR:** face embeddings are biometric data (a special category). Store them only with
  explicit consent, and delete them on request.

## Observations (fill in after running)

- Verification: AUC / EER per model.
- Open set: which threshold would you pick for a phone unlock vs. tagging holiday photos?
- What do the hardest pairs have in common (age, pose, lighting, labelling errors)?